# 1g Dynamic Braking Analysis — Jupyter Version

**Goal:** Find braking force distribution and braking power / energy distribution for a 1g deceleration event.

> Mirrors `1g dynamic.py` but as an interactive notebook with editable inputs, formatted outputs, and plots.
> All calculations refer to **front total + rear total** (both wheels per axle).

---

### How to use
1. Edit the **Inputs** cell below (masses, geometry, speeds).
2. Run all cells (`Kernel → Restart & Run All`).
3. Results and bar charts appear at the bottom.

Set `use_widgets = True` in the inputs cell for interactive sliders (requires `ipywidgets`).

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# optional: try to load widgets, fall back gracefully
try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not installed — sliders disabled, edit values directly in the next cell")

## 1 — Inputs (edit me)

Fill in measured vehicle values. Defaults use the Gen11 reference from `Brake_Calcs_Front_Rear_1g_Max_Braking.ipynb` where available.
* `total_mass` — total vehicle mass in kg
* `static_mass_front` / `static_mass_rear` — static mass per axle (must sum to total)
* `axle_distance` — wheelbase L (m)
* `cog_height` — CoG height h above ground (m)

In [ ]:
# ── Constants ──
g = 9.81  # m/s^2

initial_speed = 30.0   # m/s
final_speed = 0.0      # m/s
decel = 9.81           # m/s^2  (1g)

# ── Vehicle — EDIT THESE ──
total_mass = 272.0           # kg — None in original .py, set to 272 as in 1g max-braking notebook
static_mass_front = 177.64   # kg — derived from Wfs = (c/wb)*w  → 1743.92 N / 9.81
static_mass_rear = 94.36     # kg — derived from Wrs = 924.40 N / 9.81

axle_distance = 2.3               # m — wheelbase wb
cog_height = 19.90 * 0.0254       # m — z_cog (19.90 inch)

# toggle interactive sliders
use_widgets = False

print(f"g={g}  total_mass={total_mass} kg  front={static_mass_front} kg  rear={static_mass_rear} kg")
print(f"axle_distance={axle_distance} m  cog_height={cog_height:.4f} m  decel={decel} m/s^2")
print(f"speed: {initial_speed} → {final_speed} m/s")

In [ ]:
# ── Interactive sliders (only runs if use_widgets == True) ──
if use_widgets and HAS_WIDGETS:
    w_total = widgets.FloatSlider(value=total_mass, min=150, max=400, step=1, description='total_mass')
    w_front = widgets.FloatSlider(value=static_mass_front, min=80, max=250, step=1, description='front_mass')
    w_cog_h = widgets.FloatSlider(value=cog_height, min=0.2, max=0.8, step=0.01, description='cog_h')
    w_decel = widgets.FloatSlider(value=decel, min=2, max=12, step=0.1, description='decel')
    display(w_total, w_front, w_cog_h, w_decel)
    print("Move sliders then re-run the calculation cells below with updated values.")
    print("Tip: set static_mass_rear = total - front automatically:")
    # to actually use slider values, re-assign:
    # total_mass = w_total.value; static_mass_front = w_front.value; etc.
else:
    if use_widgets:
        print("use_widgets=True but ipywidgets not available — pip install ipywidgets")
    else:
        print("Widgets off — using direct values from previous cell.")

## 2 — Secondary Variables & Calculations

Mirrors the derivation in `1g dynamic.py`:

$$
F_{1} = \frac{W \cdot b_{rear} + m \cdot a \cdot h}{L}, \quad F_{2}=W-F_{1}
$$
where $F_1, F_2$ are dynamic normal loads (front/rear), $b_{rear}$ is CoG distance to rear axle, $L$ is wheelbase, $h$ is CoG height.
Braking and energy distributions are assumed proportional to dynamic normal loads (ideal, no lockup).

In [ ]:
# ── validation ──
assert total_mass is not None and static_mass_front is not None and static_mass_rear is not None, "Fill in masses!"
assert axle_distance is not None and cog_height is not None, "Fill in geometry!"
assert abs((static_mass_front + static_mass_rear) - total_mass) < 1.0, (
    f"Static masses {static_mass_front}+{static_mass_rear} != total {total_mass}"
)

# ── secondary variables ──
total_weight = total_mass * g
static_weight_front = static_mass_front * g
static_weight_rear = static_mass_rear * g

# NOTE: original 1g dynamic.py defines front_to_cog = L * static_mass_front / total_mass
# which is actually distance to REAR (see continuous downhill.py comment). We keep both forms:
front_to_cog_original = axle_distance * static_mass_front / total_mass  # as in 1g dynamic.py
rear_to_cog_original  = axle_distance - front_to_cog_original

# corrected (front axle → CoG uses rear mass):
front_to_cog = axle_distance * static_mass_rear / total_mass
rear_to_cog  = axle_distance - front_to_cog  # == axle_distance * static_mass_front / total_mass

energy_change = 0.5 * total_mass * (initial_speed**2 - final_speed**2)  # J
braking_force_total = total_mass * decel  # N

print(f"total_weight = {total_weight:.1f} N")
print(f"front_to_cog (corrected) = {front_to_cog:.3f} m, rear_to_cog = {rear_to_cog:.3f} m")
print(f"front_to_cog (original file) = {front_to_cog_original:.3f} m  — swapped if comparing")
print(f"energy_change = {energy_change:,.1f} J  ({energy_change/1000:.1f} kJ)")
print(f"braking_force_total = {braking_force_total:,.1f} N")

In [ ]:
# ── dynamic weight transfer (moment about rear axle) ──
dynamic_weight_front = (total_weight * rear_to_cog + total_mass * decel * cog_height) / axle_distance
dynamic_weight_rear  = total_weight - dynamic_weight_front

dynamic_braking_force_front = braking_force_total * dynamic_weight_front / total_weight
dynamic_braking_force_rear  = braking_force_total - dynamic_braking_force_front

heat_front = energy_change * dynamic_weight_front / total_weight
heat_rear  = energy_change - heat_front

# per-wheel (2 wheels per axle)
heat_front_per_rotor = heat_front / 2
heat_rear_per_rotor  = heat_rear / 2

print("── Dynamic loads (1g braking) ──")
print(f"dynamic_weight_front = {dynamic_weight_front:,.1f} N  ({dynamic_weight_front/total_weight*100:.1f}%)")
print(f"dynamic_weight_rear  = {dynamic_weight_rear:,.1f} N  ({dynamic_weight_rear/total_weight*100:.1f}%)")
print(f"sum check = {dynamic_weight_front+dynamic_weight_rear:,.1f} N vs total {total_weight:,.1f} N")
print()
print(f"dynamic_braking_force_front = {dynamic_braking_force_front:,.1f} N  ({dynamic_braking_force_front/braking_force_total*100:.1f}%)")
print(f"dynamic_braking_force_rear  = {dynamic_braking_force_rear:,.1f} N  ({dynamic_braking_force_rear/braking_force_total*100:.1f}%)")
print()
print(f"heat_front = {heat_front:,.1f} J  ({heat_front/1000:.1f} kJ)  — {heat_front_per_rotor:,.1f} J per front rotor")
print(f"heat_rear  = {heat_rear:,.1f} J  ({heat_rear/1000:.1f} kJ)  — {heat_rear_per_rotor:,.1f} J per rear rotor")
print(f"heat_total = {heat_front+heat_rear:,.1f} J")

## 3 — Visualisation

In [ ]:
labels = ['Front axle', 'Rear axle']
weights = [dynamic_weight_front, dynamic_weight_rear]
forces  = [dynamic_braking_force_front, dynamic_braking_force_rear]
heats   = [heat_front, heat_rear]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].bar(labels, weights, color=['#1f77b4','#ff7f0e'])
axes[0].set_title('Dynamic Normal Load (N)')
axes[0].set_ylabel('N')
for i, v in enumerate(weights):
    axes[0].text(i, v, f"{v:,.0f}\n({v/total_weight*100:.1f}%)", ha='center', va='bottom')

axes[1].bar(labels, forces, color=['#1f77b4','#ff7f0e'])
axes[1].set_title('Braking Force Distribution (N)')
for i, v in enumerate(forces):
    axes[1].text(i, v, f"{v:,.0f}\n({v/braking_force_total*100:.1f}%)", ha='center', va='bottom')

axes[2].bar(labels, heats, color=['#1f77b4','#ff7f0e'])
axes[2].set_title('Energy / Heat Distribution (J)')
for i, v in enumerate(heats):
    axes[2].text(i, v, f"{v/1000:.1f} kJ\n({v/energy_change*100:.1f}%)", ha='center', va='bottom')

fig.suptitle(f'1g Dynamic — {decel/g:.2f}g decel,  h={cog_height:.3f}m  L={axle_distance:.2f}m', fontsize=11)
plt.tight_layout()
plt.show()

# also a quick stopping distance / time estimate
stopping_time = (initial_speed - final_speed) / decel if decel != 0 else float('nan')
stopping_distance = (initial_speed**2 - final_speed**2) / (2*decel) if decel != 0 else float('nan')
print(f"Stopping time: {stopping_time:.2f} s")
print(f"Stopping distance: {stopping_distance:.1f} m")
print(f"Avg braking power: {energy_change/stopping_time:,.0f} W  ({energy_change/stopping_time/1000:.1f} kW)") if stopping_time else None

## 4 — Sensitivity: CoG Height Sweep

Shows how front bias increases with higher CoG (common trade-off study).

In [ ]:
h_vals = np.linspace(0.2, 0.7, 50)
front_frac = []
for h in h_vals:
    N1 = (total_weight * rear_to_cog + total_mass * decel * h) / axle_distance
    front_frac.append(N1 / total_weight * 100)

plt.figure(figsize=(7,4))
plt.plot(h_vals, front_frac, label='Front %')
plt.axvline(cog_height, color='red', linestyle='--', label=f'current h={cog_height:.3f}m')
plt.axhline(50, color='gray', linestyle=':', linewidth=0.8)
plt.xlabel('CoG height h (m)')
plt.ylabel('Front dynamic load / total (%)')
plt.title('Front bias vs CoG height — 1g braking')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---
*Notebook generated from `1g dynamic.py`. Original derivation preserved verbatim in the code comments.*